# Notebook 03a — Heuristic Attribution over User Journeys

🎯 **Objective**  
Allocate **1.0 conversion credit per converted journey** across its touchpoints using standard **heuristic attribution rules**:
- **Last-Touch**
- **First-Touch**
- **Linear** (optional de-dup inside a journey)
- **Time-Decay** (configurable `DECAY`, default `0.5`)

**Outcome:** a per-rule, **campaign-level** attribution table that feeds BI dashboards and informs budget allocation. Extra guards ensure **no double-counting** and **consistent totals**.

---

## 🧩 Inputs & Dependencies

- **Journeys (converted only)** from Notebook 02  
  - Path: `../data/processed/journeys_converted.parquet`  
  - Columns: `uid`, `conversion_id`, `campaign_seq`, `click_seq` *(list-like)*

- **Spend (for ROAS/CPA)** from impression-level logs  
  - Path: `../data/interim/pcb_dataset.parquet`  
  - Columns: `campaign`, `cost`

> This notebook **deduplicates** so each `(uid, conversion_id)` appears **once**.

---

## ⚙️ Parameters & Toggles

- `DECAY = 0.5` → strength of time-decay (closer to conversion = higher weight)  
- `CLICKS_ONLY = False` → if `True`, credit only click positions *(fallback to all if none)*  
- `DEDUP = False` → if `True`, collapse repeated campaigns *within a journey* for linear/decay

> Use these to present **click-assisted**, **position-sensitive**, or **unique-touch** narratives depending on stakeholder preference.

---

## 🧠 Attribution Rules

- **Last-Touch**: 100% credit to the last touch *(or last click if `CLICKS_ONLY=True`)*  
- **First-Touch**: 100% credit to the first touch *(or first click if `CLICKS_ONLY=True`)*  
- **Linear**: equal split across touches *(or across unique campaign IDs if `DEDUP=True`)*  
- **Time-Decay**: weight ∝ `DECAY ** (distance from conversion)`; normalized to sum to 1  
  *(optionally on clicks only; optionally deduped)*

All rules emit **journey-level credits** that sum to **1.0**.

---

## ✅ Data Hygiene & FIXES

- **Pre-list de-duplication**: drop duplicate journey rows by `["uid", "conversion_id"]`  
  - Removed: **367,466** duplicates
- **Robust list coercion** for `campaign_seq` / `click_seq` (handles JSON strings, arrays, nulls)
- **Hard integrity assertions**:
  - Per-(journey, rule) **sum(credit) = 1.0**
  - Per-rule **total credit = #converted journeys**

---

## 🔄 Processing Flow

1. **Load** converted journeys → de-dup `(uid, conversion_id)`  
2. **Apply rules** per journey → explode to long format:
3. **Integrity checks**
- 0 rows failing the `sum(credit)=1.0` check  
- Rule totals = **#converted journeys**
4. **Aggregate by (rule, campaign)** → `campaign_attribution_rules.csv`
5. **Join spend** to compute planning KPIs:
- `ROAS_attr = attributed_conversions / spend`
- `CPA_attr  = spend / attributed_conversions`
6. **Shares & Tiers** for slides:
- `spend_share`, `conv_share`, `lift_gap = conv_share − spend_share`
- Quartile **tiers** by ROAS_attr; suggest **Scale / Hold / Trim**
7. **Assist vs Closer** role:
- `assist_index = (FirstTouch − LastTouch) / (FirstTouch + LastTouch)`
- Bucket: **assist-heavy / balanced / closer-heavy** (±0.20)
8. **Recommendations**:
- Vote across rules (Scale−Trim) → **final_action**
- Merge preferred KPIs (defaults to **time-decay** metrics)

---

## 📊 Run Summary (indicative)

- **Converted journeys** used: **438,730**  
- **Exploded credits**: **1,987,502** rows  
- **Per-journey credit**: all rows sum to **1.0** ✅  
- **Per-rule total credit**: equals **438,730** for each rule ✅

*Leaderboards (Top ROAS / Lowest CPA) are exported for all rules; recurring strong IDs include:* `2869134`, `3073305`, `29531983`, `3073303`, `2073319` *(as observed in this run). Use CSVs as the source of truth.*

---

## 📂 Outputs

- `../artifacts/bi_exports/campaign_attribution_rules.csv`  
*(rule, campaign, **credit**)*
- `../artifacts/bi_exports/campaign_attribution_performance.csv`  
*(rule, campaign, **attributed_conversions**, spend, **ROAS_attr**, **CPA_attr**)*
- `../artifacts/bi_exports/campaign_attribution_tiers.csv`  
*(adds spend/conv shares, **lift_gap**, ROAS quartile **tier**, suggested **action**)*
- `../artifacts/bi_exports/campaign_recommendations.csv`  
*(created if missing: vote-based **final_action** + preferred KPIs)*
- `../artifacts/bi_exports/campaign_assist_vs_closer.csv`  
*(assist_index, funnel_role from First vs Last Touch)*
- `../artifacts/bi_exports/campaign_recommendations_with_assist.csv`  
*(recommendations enriched with assist/closer role)*

---

## 📈 Reading the Tables

- **Attribution (rules)** → “How much conversion credit did this campaign earn under rule X?”  
- **ROAS_attr / CPA_attr** → Treat attributed conversions as return to gauge efficiency.  
- **Lift vs spend** → `lift_gap > 0` converts **above** spend share (good); `< 0` underperforms.  
- **Assist vs Closer** → Use to design **sequencing** (prospecting → mid-funnel → retargeting).

---

## 🧭 Planning Guidance (for Heuristic Attribution)

- `Baseline: Use your heuristic rule (e.g., time-decay d=0.5 or last-touch) to get each campaign’s conversion_share.` 
- `Lift gap: lift_gap = conversion_share − spend_share.` 
- `Tiering: Rank campaigns by ROI and split into quartiles (Q1 best → Q4 worst).` 

**What to do**

- Scale: if tier ∈ {Q1, Q2} and lift_gap ≥ +0.01 (≥ +1 percentage point).
- Trim: if tier ∈ {Q3, Q4} and lift_gap ≤ −0.01 (≤ −1 pp).
- Hold: otherwise.

> **Caution:** If a campaign is assist-heavy (e.g., first-touch ≫ last-touch), review before trimming—it may be critical for top-of-funnel lift.

---

## 🛠️ Repro Tips & Edge Cases

- `CLICKS_ONLY=True` → click-only narratives; **fallback** to all touches if no clicks in a journey  
- `DEDUP=True` → avoid multi-counting the same campaign in linear/decay  
- Zero/NaN spend → KPIs become `NaN/Inf`; rows are preserved for manual review  
- Schema drift → `to_pylist` guards against stringified lists and nulls

---

## 📜 Changelog (this notebook)

- Fixed duplicate journeys (**−367,466** rows)  
- Enforced **sum-to-1** per (journey, rule) and **rule totals = #conversions**  
- Added **assist vs closer** role and **vote-based** recommendations  
- Exported **tiers**, **shares**, and **actions** for slide-ready insights

---

## 🔌 How this fits the pipeline
1. **Notebook 01 — Data Preprocessing & EDA** (This)
   Prepare impression-level data, KPIs, and Parquet artifacts.
2. **Notebook 02 — Build User Journeys**  
   Convert impressions → ordered user journeys with sequences.
3. **Notebook 03 — Attribution & Modeling (3a–3c)**  
   - **03a — Heuristic Attribution:** First/Last/Linear/Time-decay over journeys.(This)  
   - **03b — Journey Conversion Modeling:** Logistic Regression & XGBoost with calibration + SHAP.  
   - **03c — Shapley Attribution (Monte Carlo):** Model-based credit per campaign.
4. **Notebook 04 — Attribution Rollups, ROI & Rule-vs-Model Comparison**  
   Consolidate attributions, compute ROI, compare methods.
5. **Notebook 05 — Customer Segmentation & Audience Exports**  
   KMeans segmentation, campaign-mix profiles, and BI exports.
6. **Notebook 06 — Budget Reallocation Simulator (Upcoming)**  
   What-if scenarios using attributed ROI and constraints.

---


In [33]:

# Attribution over journeys (with FIXES)
import os
import ast
import numpy as np
import pandas as pd
from IPython.display import display

# Params
DECAY       = 0.5   # time-decay strength
CLICKS_ONLY = False # credit only click positions (falls back to all if no clicks)
DEDUP       = False # de-duplicate campaigns inside a journey for linear/decay

# Load journeys

journeys_path = "../data/processed/journeys_converted.parquet"  
if journeys_path.endswith(".parquet"):
    J = pd.read_parquet(journeys_path)
else:
    J = pd.read_csv(journeys_path)

# Keep only converted journeys if present
if "conversion" in J.columns:
    J = J[J["conversion"] == 1].copy()

# Deduplicate journeys BEFORE list coercion

dups = J.duplicated(subset=["uid","conversion_id"]).sum()
print("Duplicate journey rows:", dups)
J = J.drop_duplicates(subset=["uid","conversion_id"]).copy()

# Helpers

def to_pylist(x):
    if isinstance(x, list):
        return x
    if isinstance(x, (np.ndarray, pd.Series)):
        return x.tolist()
    if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]"):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "":
        return []
    return [x]

for col in ['campaign_seq','click_seq']:
    if col not in J.columns:
        J[col] = [[] for _ in range(len(J))]
    else:
        J[col] = J[col].apply(to_pylist)


# Attribution rule functions

def last_touch(campaign_seq, click_seq=None, clicks_only=False):
    if campaign_seq is None or len(campaign_seq) == 0:
        return None
    if clicks_only and (click_seq is not None) and (len(click_seq) > 0):
        for i in range(len(campaign_seq)-1, -1, -1):
            if i < len(click_seq) and int(click_seq[i]) == 1:
                return campaign_seq[i]
    return campaign_seq[-1]

def first_touch(campaign_seq, click_seq=None, clicks_only=False):
    if campaign_seq is None or len(campaign_seq) == 0:
        return None
    if clicks_only and (click_seq is not None) and (len(click_seq) > 0):
        for i in range(len(campaign_seq)):
            if i < len(click_seq) and int(click_seq[i]) == 1:
                return campaign_seq[i]
    return campaign_seq[0]

def linear(campaign_seq, deduplicate=False):
    if campaign_seq is None or len(campaign_seq) == 0:
        return {}
    seq = campaign_seq if not deduplicate else list(dict.fromkeys(campaign_seq))
    w = 1.0 / len(seq)
    out = {}
    for c in seq:
        out[c] = out.get(c, 0.0) + w
    return out

def time_decay(campaign_seq, click_seq=None, decay=0.5, clicks_only=False, deduplicate=False):
    n = 0 if (campaign_seq is None) else len(campaign_seq)
    if n == 0:
        return {}
    # choose positions to credit
    if clicks_only and (click_seq is not None) and (len(click_seq) > 0):
        valid_idx = [i for i in range(n) if i < len(click_seq) and int(click_seq[i]) == 1]
        if len(valid_idx) == 0:
            valid_idx = list(range(n))  # fallback to all views if no clicks
    else:
        valid_idx = list(range(n))
    # positional decay (closer to end gets more)
    raw = [(i, decay ** (n-1-i)) for i in valid_idx]
    total = sum(w for _, w in raw)
    if total == 0:
        return {}
    # normalize & fold to campaigns
    out = {}
    for i, w in raw:
        c = campaign_seq[i]
        out[c] = out.get(c, 0.0) + (w / total)
    if deduplicate:
        s = sum(out.values())
        if s > 0:
            out = {k: v/s for k, v in out.items()}
    return out


# Apply rules per journey

rows = []
for r in J.itertuples(index=False):
    uid = getattr(r, 'uid')
    conv_id = getattr(r, 'conversion_id')
    cseq = to_pylist(getattr(r, 'campaign_seq'))
    cliseq = to_pylist(getattr(r, 'click_seq'))

    # 1) Last-touch
    c = last_touch(cseq, cliseq, clicks_only=CLICKS_ONLY)
    if c is not None:
        rows.append({"uid":uid,"conversion_id":conv_id,"campaign":c,"credit":1.0,"rule":"last_touch"})

    # 2) First-touch
    c = first_touch(cseq, cliseq, clicks_only=CLICKS_ONLY)
    if c is not None:
        rows.append({"uid":uid,"conversion_id":conv_id,"campaign":c,"credit":1.0,"rule":"first_touch"})

    # 3) Linear
    d = linear(cseq, deduplicate=DEDUP)
    for camp, w in d.items():
        rows.append({"uid":uid,"conversion_id":conv_id,"campaign":camp,"credit":w,"rule":f"linear{'_dedup' if DEDUP else ''}"})

    # 4) Time-decay
    d = time_decay(cseq, cliseq, decay=DECAY, clicks_only=CLICKS_ONLY, deduplicate=DEDUP)
    for camp, w in d.items():
        rows.append({"uid":uid,"conversion_id":conv_id,"campaign":camp,"credit":w,"rule":f"time_decay_d{DECAY}{'_clicks' if CLICKS_ONLY else ''}{'_dedup' if DEDUP else ''}"})

credits_long = pd.DataFrame(rows)
print("Exploded credits shape:", credits_long.shape)
display(credits_long.head())



# Integrity check per JOURNEY (uid, conversion_id, rule)
check = (credits_long
         .groupby(['uid','conversion_id','rule'], as_index=False)['credit']
         .sum())

bad = check.loc[~np.isclose(check['credit'], 1.0, atol=1e-6)]
print(f"{len(bad)} journey-rule rows do NOT sum to 1.0 (expected 0).")
if len(bad) > 0:
    display(bad.head())
# hard guard against silent double-counts
assert len(bad) == 0, "Per-journey credit must sum to 1.0 per (uid, conversion_id, rule)"

print("\nPer-journey credit summary (should center at 1.0):")
print(check['credit'].describe())


# Campaign-level aggregates

campaign_credit = (credits_long
                   .groupby(['rule','campaign'], as_index=False)['credit']
                   .sum()
                   .sort_values(['rule','credit'], ascending=[True, False]))
display(campaign_credit.head(10))

# Save campaign-level credits
os.makedirs("../artifacts/bi_exports", exist_ok=True)
out_path = "../artifacts/bi_exports/campaign_attribution_rules.csv"
campaign_credit.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")


# Final sanity: rule totals must equal #converted journeys

n_conv = J[['uid','conversion_id']].drop_duplicates().shape[0]
rule_totals = campaign_credit.groupby('rule')['credit'].sum().round(6)
print("Converted journeys:", n_conv)
print("Rule totals:\n", rule_totals)
assert all(np.isclose(rule_totals.values, n_conv)), "Rule totals must equal #converted journeys"


Duplicate journey rows: 367466
Exploded credits shape: (1987502, 5)


,uid,conversion_id,campaign,credit,rule
0,10026553,2121096,12121528,1.000000,last_touch
1,10026553,2121096,18436526,1.000000,first_touch
2,10026553,2121096,18436526,0.600000,linear
3,10026553,2121096,12121528,0.400000,linear
4,10026553,2121096,18436526,0.419355,time_decay_d0.5


0 journey-rule rows do NOT sum to 1.0 (expected 0).

Per-journey credit summary (should center at 1.0):
count    1.754920e+06
mean     1.000000e+00
std      1.869785e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: credit, dtype: float64


,rule,campaign,credit
7,first_touch,10341182,20800.0
122,first_touch,15184511,18956.0
501,first_touch,32368244,17088.0
664,first_touch,9100693,16636.0
471,first_touch,30801593,11324.0
125,first_touch,15398570,9245.0
400,first_touch,2869134,8761.0
552,first_touch,5061834,7364.0
562,first_touch,5544859,7126.0
393,first_touch,28351001,6717.0


Saved -> ../artifacts/bi_exports/campaign_attribution_rules.csv
Converted journeys: 438730
Rule totals:
 rule
first_touch        438730.0
last_touch         438730.0
linear             438730.0
time_decay_d0.5    438730.0
Name: credit, dtype: float64


In [11]:
# Build spend per campaign from impressions parquet
import pandas as pd
import numpy as np

# Load
impr = pd.read_parquet("../data/interim/pcb_dataset.parquet", columns=["campaign", "cost"])

# Spend = sum of impression-level cost by campaign
spend = (impr.groupby("campaign", observed=True)["cost"]
              .sum()
              .reset_index()
              .rename(columns={"cost": "spend"}))

# Join with attribution credits (from the previous cell: campaign_credit = [rule, campaign, credit])
perf = (campaign_credit.rename(columns={"credit": "attributed_conversions"})
        .merge(spend, on="campaign", how="left")
        .assign(
            ROAS_attr=lambda d: d["attributed_conversions"] / d["spend"].replace(0, np.nan),
            CPA_attr =lambda d: d["spend"] / d["attributed_conversions"].replace(0, np.nan)
        ))

# Save for BI
out_perf = "../artifacts/bi_exports/campaign_attribution_performance.csv"
perf.to_csv(out_perf, index=False)
print(f"Saved -> {out_perf}")

# quick sanity
print("\nNull/zero spend rows (cannot compute ROAS_attr/CPA_attr):")
display(perf[perf["spend"].isna() | (perf["spend"] == 0)].head())


Saved -> ../artifacts/bi_exports/campaign_attribution_performance.csv

Null/zero spend rows (cannot compute ROAS_attr/CPA_attr):


,rule,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr


1) Leaderboards (top ROAS_attr / lowest CPA_attr) by rule

In [13]:
import pandas as pd

perf = pd.read_csv("../artifacts/bi_exports/campaign_attribution_performance.csv")

for rule in sorted(perf['rule'].unique()):
    print(f"\n=== {rule}: Top 10 by ROAS_attr (higher is better) ===")
    display(perf[perf['rule']==rule]
            .sort_values('ROAS_attr', ascending=False)
            .head(10)[['campaign','attributed_conversions','spend','ROAS_attr','CPA_attr']])

    print(f"\n=== {rule}: Top 10 by lowest CPA_attr (lower is better) ===")
    display(perf[perf['rule']==rule]
            .sort_values('CPA_attr', ascending=True)
            .head(10)[['campaign','attributed_conversions','spend','ROAS_attr','CPA_attr']])



=== first_touch: Top 10 by ROAS_attr (higher is better) ===


,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr
201,5871888,680.0,0.223140,3047.414162,0.000328
127,3073305,1248.0,0.421066,2963.907042,0.000337
12,2869134,9894.0,3.441929,2874.551183,0.000348
274,2073319,408.0,0.142800,2857.135934,0.000350
248,3073303,486.0,0.173583,2799.809966,0.000357
104,29531983,1564.0,0.560173,2791.992364,0.000358
162,3073302,919.0,0.353643,2598.662600,0.000385
49,30405203,3156.0,1.315260,2399.526343,0.000417
420,30487330,203.0,0.084622,2398.893765,0.000417
674,13442456,5.0,0.002094,2388.178353,0.000419



=== first_touch: Top 10 by lowest CPA_attr (lower is better) ===


,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr
201,5871888,680.0,0.223140,3047.414162,0.000328
127,3073305,1248.0,0.421066,2963.907042,0.000337
12,2869134,9894.0,3.441929,2874.551183,0.000348
274,2073319,408.0,0.142800,2857.135934,0.000350
248,3073303,486.0,0.173583,2799.809966,0.000357
104,29531983,1564.0,0.560173,2791.992364,0.000358
162,3073302,919.0,0.353643,2598.662600,0.000385
49,30405203,3156.0,1.315260,2399.526343,0.000417
420,30487330,203.0,0.084622,2398.893765,0.000417
674,13442456,5.0,0.002094,2388.178353,0.000419



=== last_touch: Top 10 by ROAS_attr (higher is better) ===


,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr
683,2869134,11919.0,3.441929,3462.884126,0.000289
785,3073305,1454.0,0.421066,3453.141698,0.000290
758,29531983,1867.0,0.560173,3332.896255,0.000300
895,3073303,524.0,0.173583,3018.725149,0.000331
923,2073319,426.0,0.142800,2983.186049,0.000335
824,3073302,973.0,0.353643,2751.358771,0.000363
879,5871888,603.0,0.223140,2702.339323,0.000370
982,29531976,299.0,0.113727,2629.105534,0.000380
1053,30487330,211.0,0.084622,2493.431450,0.000401
725,30405203,3254.0,1.315260,2474.036350,0.000404


... (6 outputs left out)


=== time_decay_d0.5: Top 10 by ROAS_attr (higher is better) ===


,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr
2147,3073305,1365.824886,0.421066,3243.732371,0.000308
2033,2869134,11091.154496,3.441929,3222.366209,0.000310
2117,29531983,1751.010262,0.560173,3125.835857,0.000320
2253,3073303,511.629096,0.173583,2947.457287,0.000339
2281,2073319,419.047491,0.142800,2934.499129,0.000341
2226,5871888,627.040986,0.223140,2810.078796,0.000356
2180,3073302,951.596018,0.353643,2690.834583,0.000372
2350,29531976,285.567340,0.113727,2510.992219,0.000398
2416,30487330,207.967907,0.084622,2457.600573,0.000407
2074,30405203,3217.312093,1.315260,2446.142307,0.000409



=== time_decay_d0.5: Top 10 by lowest CPA_attr (lower is better) ===


,campaign,attributed_conversions,spend,ROAS_attr,CPA_attr
2147,3073305,1365.824886,0.421066,3243.732371,0.000308
2033,2869134,11091.154496,3.441929,3222.366209,0.000310
2117,29531983,1751.010262,0.560173,3125.835857,0.000320
2253,3073303,511.629096,0.173583,2947.457287,0.000339
2281,2073319,419.047491,0.142800,2934.499129,0.000341
2226,5871888,627.040986,0.223140,2810.078796,0.000356
2180,3073302,951.596018,0.353643,2690.834583,0.000372
2350,29531976,285.567340,0.113727,2510.992219,0.000398
2416,30487330,207.967907,0.084622,2457.600573,0.000407
2074,30405203,3217.312093,1.315260,2446.142307,0.000409


2) Shares & tiers (useful for budget reallocation slides)

In [20]:
import pandas as pd
import numpy as np
import os

# Load attribution performance table from the previous step
perf = pd.read_csv("../artifacts/bi_exports/campaign_attribution_performance.csv")

def add_shares(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["spend_share"] = out["spend"] / out["spend"].sum()
    out["conv_share"]  = out["attributed_conversions"] / out["attributed_conversions"].sum()
    # lift_gap > 0 means campaign converts above its spend share
    out["lift_gap"] = out["conv_share"] - out["spend_share"]
    return out

def label_action(row, gap=0.01):
    # Simple guidance for slides; tweak thresholds as you like
    if row["lift_gap"] >= gap and row["tier"] in ("Q1 (top)", "Q2"):
        return "Scale"
    if row["lift_gap"] <= -gap and row["tier"] in ("Q3", "Q4 (bottom)"):
        return "Trim"
    return "Hold"

ranked_all = []
for rule in perf["rule"].unique():
    sub = perf[perf["rule"] == rule].copy()
    sub = add_shares(sub)

    # Rank by ROAS_attr (higher is better), then split into quartiles.
    # Safeguard: if there are <4 unique ranks, qcut will drop bins.
    sub["roas_rank"] = sub["ROAS_attr"].rank(method="first", ascending=False)
    n_unique = sub["roas_rank"].nunique()
    q = min(4, n_unique)
    labels = ["Q1 (top)", "Q2", "Q3", "Q4 (bottom)"][:q]

    sub["tier"] = pd.qcut(
        sub["roas_rank"], q=q, labels=labels, duplicates="drop"
    )

    sub["action"] = sub.apply(label_action, axis=1)
    sub["rule"] = rule  # keep explicit
    ranked_all.append(sub)

ranked = pd.concat(ranked_all, ignore_index=True)

# Save full table for BI
os.makedirs("../artifacts/bi_exports", exist_ok=True)
tiers_path = "../artifacts/bi_exports/campaign_attribution_tiers.csv"
ranked.to_csv(tiers_path, index=False)
print(f"Saved -> {tiers_path}")

# Quick views
for rule in ranked["rule"].unique():
    print(f"\n=== {rule}: Scale candidates (top ROAS tier & positive lift) ===")
    view = (ranked[(ranked["rule"] == rule) & (ranked["action"] == "Scale")]
            .sort_values(["tier","lift_gap"], ascending=[True, False])
            .head(15)[["campaign","ROAS_attr","CPA_attr","spend_share","conv_share","lift_gap","tier","action"]]
            .reset_index(drop=True))
    display(view)

    print(f"\n=== {rule}: Trim candidates (bottom ROAS tier & negative lift) ===")
    view = (ranked[(ranked["rule"] == rule) & (ranked["action"] == "Trim")]
            .sort_values(["tier","lift_gap"], ascending=[False, True])
            .head(15)[["campaign","ROAS_attr","CPA_attr","spend_share","conv_share","lift_gap","tier","action"]]
            .reset_index(drop=True))
    display(view)


Saved -> ../artifacts/bi_exports/campaign_attribution_tiers.csv

=== first_touch: Scale candidates (top ROAS tier & positive lift) ===


,campaign,ROAS_attr,CPA_attr,spend_share,conv_share,lift_gap,tier,action
0,10341182,314.474176,0.003180,0.028816,0.054283,0.025467,Q1 (top),Scale
1,9100693,979.905372,0.001021,0.004315,0.025331,0.021016,Q1 (top),Scale
2,30801593,800.349927,0.001249,0.005300,0.025411,0.020111,Q1 (top),Scale
3,32368244,243.974463,0.004099,0.028595,0.041791,0.013196,Q1 (top),Scale
4,15184511,231.683017,0.004316,0.031852,0.044206,0.012354,Q1 (top),Scale
5,2869134,2874.551183,0.000348,0.000713,0.012272,0.011560,Q1 (top),Scale
6,5544859,1416.670810,0.000706,0.001375,0.011668,0.010293,Q1 (top),Scale



=== first_touch: Trim candidates (bottom ROAS tier & negative lift) ===


,campaign,ROAS_attr,CPA_attr,spend_share,conv_share,lift_gap,tier,action
0,5061834,77.305711,0.012936,0.029247,0.013544,-0.015703,Q4 (bottom),Trim
1,30491418,71.700050,0.013947,0.024652,0.010588,-0.014064,Q4 (bottom),Trim
2,29427842,97.389272,0.010268,0.027145,0.015836,-0.011309,Q3,Trim



=== last_touch: Scale candidates (top ROAS tier & positive lift) ===


,campaign,ROAS_attr,CPA_attr,spend_share,conv_share,lift_gap,tier,action
0,10341182,326.805087,0.003060,0.028816,0.056412,0.027596,Q1 (top),Scale
1,9100693,1104.132921,0.000906,0.004315,0.028543,0.024227,Q1 (top),Scale
2,30801593,805.506922,0.001241,0.005300,0.025574,0.020274,Q1 (top),Scale
3,32368244,265.307384,0.003769,0.028595,0.045446,0.016850,Q1 (top),Scale
4,2869134,3462.884126,0.000289,0.000713,0.014784,0.014072,Q1 (top),Scale
5,15184511,240.498140,0.004158,0.031852,0.045888,0.014036,Q1 (top),Scale
6,5544859,1480.373558,0.000676,0.001375,0.012193,0.010818,Q1 (top),Scale
7,25920690,846.182585,0.001182,0.002640,0.013381,0.010741,Q1 (top),Scale
8,16184517,1196.087475,0.000836,0.001685,0.012074,0.010389,Q1 (top),Scale


... (6 outputs left out)


=== time_decay_d0.5: Scale candidates (top ROAS tier & positive lift) ===


,campaign,ROAS_attr,CPA_attr,spend_share,conv_share,lift_gap,tier,action
0,10341182,321.761886,0.003108,0.028816,0.055541,0.026725,Q1 (top),Scale
1,9100693,1055.569820,0.000947,0.004315,0.027287,0.022972,Q1 (top),Scale
2,30801593,799.629111,0.001251,0.005300,0.025388,0.020088,Q1 (top),Scale
3,32368244,256.645418,0.003896,0.028595,0.043962,0.015366,Q1 (top),Scale
4,15184511,236.890265,0.004221,0.031852,0.045200,0.013347,Q1 (top),Scale
5,2869134,3222.366209,0.000310,0.000713,0.013757,0.013045,Q1 (top),Scale
6,5544859,1456.380362,0.000687,0.001375,0.011995,0.010620,Q1 (top),Scale
7,25920690,820.927444,0.001218,0.002640,0.012982,0.010342,Q1 (top),Scale



=== time_decay_d0.5: Trim candidates (bottom ROAS tier & negative lift) ===


,campaign,ROAS_attr,CPA_attr,spend_share,conv_share,lift_gap,tier,action
0,30491418,69.923757,0.014301,0.024652,0.010326,-0.014326,Q4 (bottom),Trim
1,5061834,72.513763,0.013790,0.029247,0.012704,-0.016543,Q3,Trim
2,29427842,102.362661,0.009769,0.027145,0.016645,-0.010500,Q3,Trim


3) Assist vs closer signal (First-Touch vs Last-Touch) -

In [24]:
# Build recommendations, add Assist vs Closer 
import pandas as pd, numpy as np, os

ART_DIR = "../artifacts/bi_exports"
os.makedirs(ART_DIR, exist_ok=True)

tiers_path = f"{ART_DIR}/campaign_attribution_tiers.csv"
perf_path  = f"{ART_DIR}/campaign_attribution_performance.csv"
reco_path  = f"{ART_DIR}/campaign_recommendations.csv"  # may be missing
out_assist = f"{ART_DIR}/campaign_recommendations_with_assist.csv"
assist_path = f"{ART_DIR}/campaign_assist_vs_closer.csv"

# load
tiers = pd.read_csv(tiers_path)
perf  = pd.read_csv(perf_path)

# Create recommendations 
if not os.path.exists(reco_path):
    # Count votes across rules (Scale / Trim / Hold)
    votes = (tiers.groupby(["campaign","action"])
                  .size()
                  .unstack(fill_value=0)
                  .reset_index())
    for col in ["Scale","Trim","Hold"]:
        if col not in votes.columns:
            votes[col] = 0

    votes["score"] = votes["Scale"] - votes["Trim"]

    def final_action(row):
        if row["score"] >= 2:   # 2+ more Scale than Trim
            return "Scale"
        if row["score"] <= -2:  # 2+ more Trim than Scale
            return "Trim"
        return "Hold"
    votes["final_action"] = votes.apply(final_action, axis=1)

    # Add KPIs from preferred planning rule (time-decay by default)
    rule_pref = [r for r in perf["rule"].unique() if r.startswith("time_decay")]
    rule_pref = rule_pref[0] if rule_pref else perf["rule"].iloc[0]
    kpis = perf[perf["rule"]==rule_pref][["campaign","attributed_conversions","spend","ROAS_attr","CPA_attr"]]

    reco = (votes.merge(kpis, on="campaign", how="left")
                 .sort_values(["final_action","score","ROAS_attr"], ascending=[True, False, False]))
    reco.to_csv(reco_path, index=False)
    print(f"Created -> {reco_path}")
else:
    reco = pd.read_csv(reco_path)
    print(f"Loaded  -> {reco_path}")

# Assist vs closer signal 
rules = set(perf["rule"].unique())
if {"first_touch","last_touch"} <= rules:
    wide = (perf.pivot_table(index="campaign", columns="rule",
                             values="attributed_conversions", aggfunc="sum")
              .fillna(0))
    denom = wide.get("first_touch",0) + wide.get("last_touch",0)
    wide["assist_index"] = np.where(denom > 0,
                                    (wide.get("first_touch",0) - wide.get("last_touch",0)) / denom,
                                    np.nan)

    def role(x, t=0.20):
        if pd.isna(x): return "unknown"
        if x >=  t:   return "assist-heavy"
        if x <= -t:   return "closer-heavy"
        return "balanced"
    wide["funnel_role"] = wide["assist_index"].apply(role)

    wide_out = wide[["assist_index","funnel_role"]].reset_index()
    wide_out.to_csv(assist_path, index=False)
    print(f"Saved  -> {assist_path}")

    merged = reco.merge(wide_out, on="campaign", how="left")
    merged.to_csv(out_assist, index=False)
    print(f"Saved  -> {out_assist}")

    # Optional: flag trims that are actually assist-heavy (review before cutting)
    risky = merged[(merged["final_action"]=="Trim") & (merged["assist_index"] > 0.20)]
    print(f"Assist-heavy trims to review: {len(risky)}")
    if not risky.empty:
        display(risky.sort_values("assist_index", ascending=False).head(15)[
            ["campaign","assist_index","funnel_role","final_action","ROAS_attr","CPA_attr","spend","attributed_conversions"]
        ])
else:
    print("Cannot compute assist_index: need both first_touch and last_touch rules present in perf.")


Created -> ../artifacts/bi_exports/campaign_recommendations.csv
Saved  -> ../artifacts/bi_exports/campaign_assist_vs_closer.csv
Saved  -> ../artifacts/bi_exports/campaign_recommendations_with_assist.csv
Assist-heavy trims to review: 0
